# Laboratorio Dry Bean - Sprint 3: despliegue

**Proyecto:** Clasificación de variedades de frijol seco (Dry Bean Dataset, UCI 602)
**Equipo:** Jacobo Unzueta y Rommel Prieto - Maestría en IA y Ciencia de Datos, Universidad Autónoma de Occidente
**Metodologías:** CRISP-DM · TDSP · Scrum ML
**Sprint 3:** PB-06 (despliegue) y PB-07 (documentación final)

## Objetivo del sprint

El modelo ya está entrenado y guardado. Ahora se **carga desde disco**, se verifica
que reproduce las métricas reportadas y se usa para predecir granos nuevos con una
función reutilizable.

**Requisito:** haber ejecutado antes `python -m src.sprint2` (genera
`outputs/models/modelo_final_drybean.joblib`).


## 0. Configuración e importaciones

In [1]:
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from src import config, deploy, models, prepare
from src.eda import dataframe_to_markdown

pd.set_option("display.width", 120)
print("Raíz del proyecto:", config.PROJECT_ROOT)
print("Artefacto esperado:", config.FINAL_MODEL_PATH.name)

Raíz del proyecto: C:\Users\Jacobo\Documents\GestionProyectosIA\laboratorio_drybean_ml
Artefacto esperado: modelo_final_drybean.joblib


## 1. Cargar el modelo final desde disco

`deploy.load_final_model()` lee el `.joblib` y su archivo de metadatos. Los
metadatos son la trazabilidad del artefacto: qué modelo es, con qué métricas,
qué variables espera y qué clases predice.

In [2]:
modelo, metadatos = deploy.load_final_model()

print("Modelo:", metadatos["selected_model"])
print("Tipo:", metadatos["model_type"])
print("Guardado (UTC):", metadatos["saved_at_utc"])
print("scikit-learn:", metadatos["sklearn_version"])
print("Variables:", len(metadatos["features"]))
print("Clases:", ", ".join(metadatos["classes"]))

Modelo: HistGradientBoosting
Tipo: HistGradientBoostingClassifier
Guardado (UTC): 2026-11-14T04:16:52+00:00
scikit-learn: 1.9.1
Variables: 16
Clases: BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER, SIRA


In [3]:
resumen = pd.DataFrame(
    {
        "Métrica": ["Accuracy (test)", "F1 macro (test)", "F1 macro CV 5-fold"],
        "Valor": [
            f"{metadatos['metrics']['accuracy']:.4f}",
            f"{metadatos['metrics']['f1_macro']:.4f}",
            f"{metadatos['metrics']['cv_f1_macro_mean']:.4f} ± {metadatos['metrics']['cv_f1_macro_std']:.4f}",
        ],
    }
)
resumen.index = range(1, len(resumen) + 1)
resumen.index.name = "#"
print(dataframe_to_markdown(resumen))

| # | Métrica | Valor |
|---|---|---|
| 1 | Accuracy (test) | 0.9195 |
| 2 | F1 macro (test) | 0.9315 |
| 3 | F1 macro CV 5-fold | 0.9429 ± 0.0032 |


**Pregunta del laboratorio:** ¿por qué guardar metadatos junto al modelo?

**Respuesta:** un `.joblib` es una caja negra. Sin metadatos no se sabe qué
variables espera, en qué orden, qué clases puede devolver ni con qué métricas se
validó. Los metadatos permiten detectar un archivo equivocado antes de predecir.

## 2. Verificación de integridad

Se recargan los datos de prueba y se vuelve a predecir con el modelo leído de
disco. Si las métricas no coinciden con las guardadas, el artefacto está corrupto
o fue entrenado con otros datos.

In [4]:
X_train, X_test, y_train, y_test = prepare.load_processed_data()
print(f"Test: {X_test.shape[0]} filas · {X_test.shape[1]} variables")

Test: 2709 filas · 16 variables


In [5]:
predicciones = modelo.predict(X_test)
accuracy = accuracy_score(y_test, predicciones)
f1_macro = f1_score(y_test, predicciones, average="macro")

verificacion = pd.DataFrame(
    {
        "Comprobación": ["Accuracy", "F1 macro", "Variables", "Clases"],
        "Guardado": [
            f"{metadatos['metrics']['accuracy']:.6f}",
            f"{metadatos['metrics']['f1_macro']:.6f}",
            str(len(metadatos["features"])),
            str(len(metadatos["classes"])),
        ],
        "Recalculado": [
            f"{accuracy:.6f}",
            f"{f1_macro:.6f}",
            str(len(modelo.feature_names_in_)),
            str(len(modelo.classes_)),
        ],
    }
)
verificacion.index = range(1, len(verificacion) + 1)
verificacion.index.name = "#"
print(dataframe_to_markdown(verificacion))

assert abs(accuracy - metadatos["metrics"]["accuracy"]) < 1e-9
assert abs(f1_macro - metadatos["metrics"]["f1_macro"]) < 1e-9
assert list(modelo.feature_names_in_) == list(metadatos["features"])
print("\nArtefacto integro: las metricas se reproducen exactamente.")

| # | Comprobación | Guardado | Recalculado |
|---|---|---|---|
| 1 | Accuracy | 0.919528 | 0.919528 |
| 2 | F1 macro | 0.931461 | 0.931461 |
| 3 | Variables | 16 | 16 |
| 4 | Clases | 7 | 7 |

Artefacto integro: las metricas se reproducen exactamente.


## 3. Predicción de varios granos

`predict_beans` devuelve la clase y la probabilidad de la clase elegida.

In [6]:
demo = X_test.head(8).copy()
resultados = deploy.predict_beans(modelo, demo)

tabla = pd.DataFrame(resultados)
tabla.insert(0, "clase_real", y_test.head(8).to_numpy())
tabla["acierto"] = tabla["clase_real"] == tabla["prediccion"]
tabla.index = range(1, len(tabla) + 1)
tabla.index.name = "muestra"
print(dataframe_to_markdown(tabla))

| muestra | clase_real | prediccion | probabilidad | clase_mas_probable | acierto |
|---|---|---|---|---|---|
| 1 | SEKER | SEKER | 0.997 | SEKER | True |
| 2 | DERMASON | DERMASON | 0.999 | DERMASON | True |
| 3 | DERMASON | DERMASON | 0.992 | DERMASON | True |
| 4 | HOROZ | HOROZ | 0.999 | HOROZ | True |
| 5 | SEKER | SEKER | 0.999 | SEKER | True |
| 6 | DERMASON | DERMASON | 0.717 | DERMASON | True |
| 7 | DERMASON | DERMASON | 0.977 | DERMASON | True |
| 8 | DERMASON | DERMASON | 0.999 | DERMASON | True |


## 4. Predicción de un grano nuevo (paso 18 del laboratorio)

Se simula la llegada de un grano con las 16 medidas geométricas. `predict_bean_class`
acepta un diccionario, una `Series` o una fila de `DataFrame`.

In [7]:
fila = X_test.iloc[0]
nuevo_grano = {columna: round(float(valor), 2) for columna, valor in fila.items()}
print(json.dumps(nuevo_grano, indent=4, ensure_ascii=False))

{
    "Area": 40512.0,
    "Perimeter": 737.64,
    "MajorAxisLength": 248.2,
    "MinorAxisLength": 208.53,
    "AspectRatio": 1.19,
    "Eccentricity": 0.54,
    "ConvexArea": 40973.0,
    "EquivDiameter": 227.12,
    "Extent": 0.74,
    "Solidity": 0.99,
    "Roundness": 0.94,
    "Compactness": 0.92,
    "ShapeFactor1": 0.01,
    "ShapeFactor2": 0.0,
    "ShapeFactor3": 0.84,
    "ShapeFactor4": 1.0
}


In [8]:
clase = deploy.predict_bean_class(modelo, nuevo_grano)
detalle = deploy.predict_beans(modelo, nuevo_grano)[0]

print("Clase real:     ", y_test.iloc[0])
print("Prediccion:     ", clase)
print("Probabilidad:   ", detalle["probabilidad"])
print("Clase mas probable:", detalle["clase_mas_probable"])

Clase real:      SEKER
Prediccion:      SEKER
Probabilidad:    0.8927
Clase mas probable: SEKER


## 5. Contrato de entrada: errores explicados

La función valida la entrada antes de predecir. Un error claro es mejor que un
`KeyError` de scikit-learn en producción.

In [9]:
casos = {
    "Variable faltante": {k: v for k, v in list(nuevo_grano.items())[:10]},
    "Variable desconocida": {**nuevo_grano, "Color": 3},
    "Valor nulo": {**nuevo_grano, "Area": None},
    "Texto en vez de numero": {**nuevo_grano, "Area": "grande"},
    "Entrada vacia": {},
}

for nombre, entrada in casos.items():
    try:
        deploy.predict_bean_class(modelo, entrada)
    except deploy.DatosInvalidosError as error:
        print(f"{nombre:24} -> {error}")

Variable faltante        -> Faltan variables requeridas: ['Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']. Se esperaban 16 variables geométricas.
Variable desconocida     -> Hay variables no reconocidas: ['Color']. Revisa los nombres.
Valor nulo               -> La entrada contiene valores nulos.
Texto en vez de numero   -> Las variables deben ser numéricas. Revisa: ['Area']
Entrada vacia            -> La entrada no contiene variables. Envía las medidas geométricas del grano.


## 6. El modelo cargado es equivalente al recién entrenado

Se reentrena el mismo modelo con la misma semilla y se comparan las predicciones. Sirve para confirmar que el `.joblib` no perdió información al serializarse.

In [10]:
reentrenado, _ = models.train_model(models.build_hgb_model(), X_train, y_train)
coinciden = (reentrenado.predict(X_test) == predicciones).mean()
print(f"Coincidencia entre modelo en disco y modelo reentrenado: {coinciden:.2%}")

Coincidencia entre modelo en disco y modelo reentrenado: 100.00%


## 7. Dónde falla el modelo

La verificación dice que el artefacto es correcto, no que el modelo sea perfecto.
Conviene mirar los errores antes de llevarlo a producción.

In [11]:
errores = pd.DataFrame({"real": y_test, "predicho": predicciones})
errores = errores[errores["real"] != errores["predicho"]]
pares = errores.value_counts().head(5).rename("casos").reset_index()
pares.index = range(1, len(pares) + 1)
pares.index.name = "#"
print(dataframe_to_markdown(pares))

| # | real | predicho | casos |
|---|---|---|---|
| 1 | SIRA | DERMASON | 53 |
| 2 | DERMASON | SIRA | 47 |
| 3 | BARBUNYA | CALI | 16 |
| 4 | DERMASON | SEKER | 10 |
| 5 | HOROZ | CALI | 9 |


In [12]:
reporte = classification_report(y_test, predicciones, digits=3, zero_division=0)
print(reporte)

              precision    recall  f1-score   support

    BARBUNYA      0.923     0.902     0.912       265
      BOMBAY      1.000     1.000     1.000       104
        CALI      0.922     0.945     0.933       326
    DERMASON      0.907     0.918     0.912       709
       HOROZ      0.959     0.941     0.950       372
       SEKER      0.946     0.946     0.946       406
        SIRA      0.870     0.863     0.867       527

    accuracy                          0.920      2709
   macro avg      0.932     0.931     0.931      2709
weighted avg      0.920     0.920     0.920      2709



`SIRA` y `DERMASON` concentran la mayor parte del error y `BOMBAY` se clasifica
sin fallos. Es el mismo patrón del Sprint 2: el límite está en las variedades de
tamaño y forma parecidos, no en las clases con pocos ejemplos.

**Consecuencia práctica:** para control de calidad estricto conviene revisión
humana cuando la predicción es `SIRA` o `DERMASON`, o cuando la probabilidad es baja.

## 8. Conclusiones del Sprint 3

1. El modelo se carga desde disco con `joblib` y reproduce **exactamente** las
   métricas reportadas en el Sprint 2.
2. `predict_bean_class` y `predict_beans` permiten predecir uno o varios granos
   con validación de contrato de entrada.
3. El artefacto está listo para un endpoint o un script de lote: reentrenar con
   `python -m src.sprint2` y volver a verificar con `python -m src.sprint3`.
4. Las limitaciones son del problema, no del despliegue: 16 variables geométricas,
   confusión sistemática `SIRA` ↔ `DERMASON` y un desempeño medido en una partición fija.
5. Con esto se cierra el ciclo CRISP-DM: comprensión del negocio y de los datos,
   preparación, modelado, evaluación y despliegue.
